In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import *
from statsmodels.stats import weightstats, proportion


In [2]:
data = pd.read_csv('../data/data_sessions.csv', parse_dates=['datetime', 'date'])

In [5]:
data.head(1)

,EventName,DeviceIDHash,ExpId,datetime,date,session
0,MainScreenAppear,6888746892508752,246,2019-08-06 14:06:34,2019-08-06,1


In [12]:
# количество участников в группах ААВ
#data.groupby('ExpId').agg(uniq_in_qroup = ('DeviceIDHash', 'nunique'), uniq_total = ('DeviceIDHash', lambda x: data['DeviceIDHash'].nunique())).reset_index()


,ExpId,uniq_in_qroup,uniq_total
0,246,2484,7534
1,247,2513,7534
2,248,2537,7534


Этап 1. Валидация теста (А1=246 и А2=247) 

# Проверка размера групп (корректность сплитования).
- Сравниваем количество уникальных пользователей (DeviceIDHash) в A1 и A2. Ожидаемый сплит 50/50 (0.5/0.5)
- Метод: хи-тест (chisquare)
- Критерий: уровень значимости = 0.05

# АА-сравнение метрик за период теста. 
Разобъем метрики на группы:
- Конверсии (CR_to_offer, CR_to_cart, CR_to_paid) - доля клиентов дошедших до определенного события.  
 -- Метод: z-тест для пропорций
- Интенсивность (Offers_per_user, Purchases_per_user, Sessions_per_user) - глубина взаимодействия  
 -- Метод: t-тест оценка среднего
- Эффективность (Purchases_per_session) - конверсия в покупку за одну сессию. 
 -- Метод: t-тест оценка среднего   

Так как мы сравниваем множество метрик, возможно столкнемся с ситуацией, когда нужна будет корректировка уровня значимости  
 -- Метод: более строгая Бонферрони или пошаговая корректировка Холм-Бонферрони. 


События MainScreenAppear и Tutorial можно считать ненадежными метриками, и убрать их с расчета пропрций, так как они не являются целевыми:
 - наличие Tutorial может наблюдаться как у новых, так и у существующих клиентов. Не является признаком "нового клиента"
 - наличие MainScreenAppear не является фактом захода в ПО (в ПО можно заходить и другими способоами - push, глубокие ссылки, прямые виджеты и т.д.) или обязательного этапа покупки. Событие может встречаться или отсутствовать при взаимодействии с приложением не зависимо от того, была ли совершена покупка или нет.
 - при этом для нас важно, что уменьшение множественности проверок уменьшает шанс получения ложного сигнала о невалидности теста

Но в рамках данного проекта мы пока оставим эти метрики 

При анализе А/В групп можно рассмотреть доп.метрику Purchases_per_perchasing_session - метрика мультипокупки (если сессия с покупкой, то сколько за сессию покупок)


In [34]:
# проверка корректности сплита
alpha = 0.05
amount_user_dict = data.groupby('ExpId')['DeviceIDHash'].nunique().to_dict()
observed = [amount_user_dict[246], amount_user_dict[247]]
expected = [sum(observed)*0.5, sum(observed)*0.5]
chi_pvalue = chisquare(observed, f_exp = expected).pvalue
if chi_pvalue > alpha:
    print(f'Значение p-value = {round(chi_pvalue,4)} > {alpha}. Расхождения не являются статистически значимыми. Нет оснований отвергать гипотезу о равенстве пропорций')
    print(f'Пропорции соблюдены. Сплитоварие можно считать корректным')
else:
    print(f' Значение p-value = {chi_pvalue} < {alpha}. Расхождения являются статистически значимыми. Гипотеза о равенстве пропорций должна быть отклонена в пользу альтернативной версии')
    print(f'Пропорции не соблюдены. Необходимо скорректировать тест.')


Значение p-value = 0.6816 > 0.05. Расхождения не являются статистически значимыми. Нет оснований отвергать гипотезу о равенстве пропорций
Пропорции соблюдены. Сплитоварие можно считать корректным


In [118]:
# Проверка на сопоставимость групп АА (метрики и признаки распределены одинаково между группами)

# оставим данные только по АА группам + столбец с кол-вом уникальных пользователей в каждой группе
AA_group = data.query('ExpId in [246,247]').copy()
AA_group['user_in_group'] = AA_group.groupby('ExpId')['DeviceIDHash'].transform('nunique')

pvalue_dict = {} # собираем сюда все pvalue, которые получим по каждой метрике

# Шаг 1
# Метрики Конверсии в группах (CR_to_offer, CR_to_cart, CR_to_paid) - доля клиентов дошедших до определенного события.

## группируем по событиям в каждой группе, считаем уникальных пользователей по каждому событию
events_uniq_user_in_group = (AA_group
                        .groupby(['ExpId', 'EventName'])
                        .agg(uniq_user = ('DeviceIDHash', 'nunique'),
                             user_in_group=('user_in_group', 'first'))
                        .reset_index()
)
event_name = list(events_uniq_user_in_group['EventName'].unique()) # список названий событий
## вычиляем pvalue по каждой метрике и добавляем в словарь
for event in event_name:
    observed = events_uniq_user_in_group[events_uniq_user_in_group['EventName'] == event]['uniq_user'].to_list()
    nobs = events_uniq_user_in_group[events_uniq_user_in_group['EventName'] == event]['user_in_group'].to_list() 
    z_stat, p = proportion.proportions_ztest(observed, nobs)
    pvalue_dict[event] = round(float(p),4) 

# Шаг2
# Метрики Интенсивности в группах (Offers_per_user, Purchases_per_user, Sessions_per_user)  - события на одного пользователя  
# Метрики Эффективности (Purchases_per_session) - покупок за сессию 

## по каждому пользователю считаем количество его активностей, значимых для нас (OffersScreenAppear, PaymentScreenSuccessful, session)
active_per_user_in_group = (AA_group
                            .groupby(['ExpId', 'DeviceIDHash'])
                            .agg(Offers_per_user = ('EventName',lambda x: (x == 'OffersScreenAppear').sum()),
                                Purchases_per_user = ('EventName',lambda x: (x == 'PaymentScreenSuccessful').sum()),
                                Sessions_per_user = ('session', 'nunique')
                                )
).reset_index()
# добавляем столбец Purchases_per_session
active_per_user_in_group['Purchases_per_session'] = active_per_user_in_group['Purchases_per_user'] / active_per_user_in_group['Sessions_per_user']
metrics = list(active_per_user_in_group.columns)[2:] # список названий метрик (названий столбцов)
## вычиляем pvalue по каждой метрике и добавляем в словарь
for metric in metrics:
    value_246 = active_per_user_in_group[active_per_user_in_group['ExpId'] == 246][metric]
    value_247 = active_per_user_in_group[active_per_user_in_group['ExpId'] == 247][metric]
    t_stat, t_p = ttest_ind(value_246, value_247, equal_var=False)
    pvalue_dict[metric] = round(float(t_p),4) # результаты добавляем в словарь

pvalue_dict.values()


{'CartScreenAppear': 0.2288,
 'MainScreenAppear': 0.7571,
 'OffersScreenAppear': 0.2481,
 'PaymentScreenSuccessful': 0.1146,
 'Tutorial': 0.9377,
 'Offers_per_user': 0.7777,
 'Purchases_per_user': 0.2415,
 'Sessions_per_user': 0.1772,
 'Purchases_per_session': 0.1391}

In [124]:
# Шаг3 
# Анализируем полученные pvalue всем метрик с нашим уровнем значимости 0.05
# При необходимости ставим вопрос о применении теста Холма-Бонферрони

if all(p > alpha for k in pvalue_dict.values()):
    print(f'''
Все p-value превышают установленный уровень значимости {alpha}.
Ни одна метрика не показала статистически значимых различий между группами А/А-теста.
Значит группы сопоставимы.
Тест можно считать валидным.
''')
else:
    n_metrics = len(pvalue_dict)
    expected_false_positives = n_metrics * alpha
    observed_false_positives = sum(p < alpha for p in pvalue_dict.values())
    print(f"Ожидаемое число ложноположительных: ~{expected_false_positives:.1f}")
    print(f"Наблюдаемое: {observed_false_positives}")


Все p-value превышают установленный уровень значимости 0.05.
Ни одна метрика не показала статистически значимых различий между группами А/А-теста.
Значит группы сопоставимы.
Тест можно считать валидным.

